# Build Mimi Token Dataset From A Hugging Face Audio Dataset

This notebook clones the full `kyutai-labs/moshi` repo, installs the repo dependencies, and runs `scripts/build_mimi_hf_dataset.py`.

Use a GPU runtime when possible. By default the conversion script expects these dataset columns:
- source audio: `audio_eng`
- target audio: `audio_lug`
- source text: `text_eng`
- target text: `text_lug`

The build writes a local staging directory with manifests and Mimi code tensors, then uploads the result to a Hugging Face dataset repo such as `<dataset>_mimi_token_version`.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kyutai-labs/moshi.git"
WORKSPACE_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKSPACE_ROOT / "moshi"

if REPO_DIR.exists():
    print(f"Repo already exists at {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "status", "-sb"], check=True)
print(f"Working from: {REPO_DIR}")

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "moshi" / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR / "moshi")], check=True)

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("device:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", exc)

In [ ]:
import getpass
import os

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN: ")

print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
from pathlib import Path

SOURCE_DATASET = "yigagilbert/salt-s2s-lug-eng-studio-dataset"
SOURCE_CONFIG = ""
TARGET_REPO_ID = ""
SPLITS = "all"
DEVICE = "cuda"
BATCH_SIZE = 8
NUM_CODEBOOKS = 8
PRIVATE_REPO = True

ID_COL = "id"
SRC_AUDIO_COL = "audio_eng"
TGT_AUDIO_COL = "audio_lug"
SRC_TEXT_COL = "text_eng"
TGT_TEXT_COL = "text_lug"

default_repo_id = (
    f"{SOURCE_DATASET.split('/', 1)[0]}/{SOURCE_DATASET.split('/', 1)[1]}_mimi_token_version"
    if "/" in SOURCE_DATASET
    else f"{SOURCE_DATASET}_mimi_token_version"
)
RESOLVED_REPO_ID = TARGET_REPO_ID or default_repo_id
LOCAL_OUTPUT_DIR = REPO_DIR / "artifacts" / "mimi_builds" / RESOLVED_REPO_ID.replace("/", "__")
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Source dataset:", SOURCE_DATASET)
print("Target Mimi token repo:", RESOLVED_REPO_ID)
print("Local staging dir:", LOCAL_OUTPUT_DIR)

In [ ]:
import os
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    str(REPO_DIR / "scripts" / "build_mimi_hf_dataset.py"),
    "--dataset", SOURCE_DATASET,
    "--splits", SPLITS,
    "--output", str(LOCAL_OUTPUT_DIR),
    "--repo-id", RESOLVED_REPO_ID,
    "--device", DEVICE,
    "--batch-size", str(BATCH_SIZE),
    "--num-codebooks", str(NUM_CODEBOOKS),
    "--id-col", ID_COL,
    "--src-audio-col", SRC_AUDIO_COL,
    "--tgt-audio-col", TGT_AUDIO_COL,
    "--src-text-col", SRC_TEXT_COL,
    "--tgt-text-col", TGT_TEXT_COL,
    "--hf-token", os.environ["HF_TOKEN"],
]

if SOURCE_CONFIG:
    cmd.extend(["--config", SOURCE_CONFIG])
if PRIVATE_REPO:
    cmd.append("--private")

print("Running:\n", " ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR)

In [ ]:
import json
from itertools import islice

stats_path = LOCAL_OUTPUT_DIR / "stats.json"
build_config_path = LOCAL_OUTPUT_DIR / "build_config.json"

stats = json.loads(stats_path.read_text())
build_config = json.loads(build_config_path.read_text())

print("Stats:")
print(json.dumps(stats, indent=2))
print("\nBuild config:")
print(json.dumps(build_config, indent=2))

for split in stats.get("splits", {}):
    manifest_path = LOCAL_OUTPUT_DIR / f"dataset.{split}.jsonl"
    if manifest_path.exists():
        print(f"\nPreview: {manifest_path.name}")
        with manifest_path.open("r", encoding="utf-8") as handle:
            for line in islice(handle, 3):
                print(line.strip())

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
repo_files = api.list_repo_files(repo_id=RESOLVED_REPO_ID, repo_type="dataset")
print(f"Uploaded file count: {len(repo_files)}")
for path in repo_files[:25]:
    print(path)